In [ ]:
#@markdown HiggsAudio-V2
import os, re, subprocess, threading, time, urllib.request, socket, pytz, codecs, shutil, pexpect, random, string
from IPython.display import Image, clear_output, display, HTML
from IPython.utils import capture
from pathlib import Path
import os

# Environment configurations
os.environ["TF_CUDNN_WORKSPACE_LIMIT_IN_MB"] = "4096"
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"

# Automatic environment detection
if 'COLAB_GPU' in os.environ:  # Google Colab
    base_path = "/content"
    print("Environment detected: Google Colab")
elif os.path.exists('/kaggle'):  # Kaggle
    base_path = "/kaggle/working"
    print("Environment detected: Kaggle")
elif os.path.exists('/teamspace'):  # Lightning AI
    base_path = "/teamspace/studios/this_studio"
    print("Environment detected: Lightning")
elif os.path.exists('/home'):  # SageMaker
    base_path = "/home/studio-lab-user"
    print("Environment detected: SageMaker")
else:
    print("Environment no detected, Selecting Google Colab as default.")
    base_path = "/content"

# Base paths
higgs_path = os.path.join(base_path, "higgs-audio")
higgs_local_path = os.path.join(base_path, "HiggsAudio-V2-Local")
higgs_quantized_path = os.path.join(base_path, "higgs-audio_quantized")
tmp_path = os.path.join(base_path, "tmp")

def install_dependencies():
    """Install Higgs Audio basic dependencies"""
    print("📦 Installing dependencies...")
    os.chdir(higgs_path)
    !pip install -r requirements.txt
    !pip install -e .
    !pip install bitsandbytes
    !npm install -g localtunnel
    !pip install protobuf --upgrade --force-reinstall

def start_localtunnel(port=7860):
    """Start LocalTunnel to expose the interface"""
    def tunnel_thread():
        # Wait for port to be available
        print(f"⏳ Waiting for port {port}...")
        while True:
            time.sleep(0.5)
            sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            result = sock.connect_ex(('127.0.0.1', port))
            if result == 0:
                sock.close()
                print(f"✅ Port {port} available")
                break
            sock.close()

        # Show public IP once
        try:
            ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
            print(f"🌐 Public IP: {ip}")
        except:
            print("⚠️ Could not get public IP")

        # Launch localtunnel
        print("🚀 Starting LocalTunnel...")
        try:
            p = subprocess.Popen(["lt", "--port", str(port)], 
                               stdout=subprocess.PIPE, 
                               universal_newlines=True)
            for line in p.stdout:
                line = line.strip()
                if line:
                    print(f"🔗 {line}")
        except Exception as e:
            print(f"❌ Error: {e}")

    threading.Thread(target=tunnel_thread, daemon=True).start()

def clone_repositories():
    """Clone all necessary repositories"""
    os.chdir(base_path)
    
    # Clone main repository
    if not os.path.exists(higgs_path):
        print("📥 Clonando higgs-audio...")
        !git clone https://github.com/boson-ai/higgs-audio.git
    
    # Clone quantized version
    if not os.path.exists(higgs_quantized_path):
        print("📥 Clonando higgs-audio_quantized...")
        !git clone https://github.com/Nyarlth/higgs-audio_quantized
    
    # Clone local V2 version
    if not os.path.exists(higgs_local_path):
        print("📥 Clonando HiggsAudio-V2-Local...")
        !git clone https://github.com/PierrunoYT/HiggsAudio-V2-Local.git

# Main menu
while True:
    print("")
    print("\nSelect an option:")
    print("1: Install Higgs Audio")
    print('\033[38;2;0;255;252m' + "2: >> Run Higgs Audio (Local V2) <<" + '\033[0m')
    print("3: Delete Higgs Audio")
    print("0: Exit")
    
    opcion = input("Enter your option: ")

    if opcion == "1":
        # Complete installation
        clone_repositories()
        install_dependencies()
        
        clear_output()
        print('\033[38;2;0;243;243m' + 'Higgs Audio Installation Complete 100% ✓' + '\033[0m')

    elif opcion == "2":
        # Run local V2 version (highlighted option)
        if not os.path.exists(higgs_local_path):
            print("❌ HiggsAudio-V2-Local is not installed. Use option 1 first.")
            continue
            
        os.chdir(higgs_local_path)
        start_localtunnel(7860)
        
        print("🎵 Starting HiggsAudio V2 Local...")
        !python gradio_interface.py

    elif opcion == "3":
        # Remove installation
        clear_output()
        print("")
        print("Removing folders...")

        paths_to_remove = [higgs_path, higgs_local_path, higgs_quantized_path, tmp_path]
        
        for path in paths_to_remove:
            if os.path.exists(path):
                shutil.rmtree(path)
                print(f"✅ Folder removed: {path}")
            else:
                print(f"⚠️ The folder {path} does not exist.")

        time.sleep(3)
        clear_output()

    elif opcion == "0":
        print("Exiting...")
        print('\033[38;2;0;243;243m' + 'Higgs Audio FINISHED (ㅠ﹏ㅠ)' + '\033[0m')
        clear_output()
        break

    else:
        print("Invalid option. Please try again.")
        print('\033[38;2;0;243;243m' + 'Higgs Audio FINISHED (ㅠ﹏ㅠ)' + '\033[0m')